# JobHuntAI — Component Tests

Each section is runnable independently with just a populated `.env`.
Run this from the project root so imports resolve.


## 0. Setup — load env + fix sys.path

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from dotenv import load_dotenv
load_dotenv()
print('OPENAI key set:', bool(os.getenv('OPENAI_API_KEY')))

## 1. DB Layer
Create tables, insert a user/session/messages, query them back.

In [ ]:
from db import sqlite
sqlite.init_db()
import uuid
uname = 'nb_' + uuid.uuid4().hex[:6]
user = sqlite.create_user(uname, uname + '@example.com', 'hashed-pw')
session = sqlite.create_session(user['user_id'], title='Notebook test')
sqlite.add_message(session['session_id'], 'user', 'Hello agent')
sqlite.add_message(session['session_id'], 'assistant', 'Hi! How can I help?')
print('User:', user['user_id'])
print('Messages:', sqlite.get_messages(session['session_id']))

## 2. Auth
Hash a password, issue a JWT, decode it back.

In [ ]:
from api import auth
hashed = auth.hash_password('secret123')
print('verify ok:', auth.verify_password('secret123', hashed))
tok = auth.create_access_token('user-abc')
print('token:', tok[:40], '...')
print('decoded sub:', auth.decode_token(tok))

## 3. ChromaDB
Upsert 3 docs to `memory:test_user`, query top-2.

In [ ]:
from db import chroma
ns = chroma.memory_ns('test_user')
chroma.delete_namespace(ns)
chroma.upsert(ns, documents=[
    'Senior backend engineer, 8 years Python.',
    'Led monolith to microservices migration.',
    'Prefers remote-first fintech companies.'],
    metadatas=[{'k':'skill'},{'k':'achievement'},{'k':'pref'}])
docs, metas, dists = chroma.query(ns, 'preferred company type', 2)
for d, dist in zip(docs, dists): print(round(dist,3), d)

## 4. Embeddings
Load all-MiniLM-L6-v2 and embed a sentence.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
vec = model.encode('A backend engineer who loves distributed systems.')
print('vector shape:', vec.shape)

## 5. Resume Ingestion (structured extraction)
Run GPT-4o extraction on a sample resume text string.

In [ ]:
import asyncio
from agent.llm import complete_json
from agent.resume.ingestion import _EXTRACTION_SYSTEM
SAMPLE = '''Jane Doe | jane@x.com | 555-1234 | NYC
Summary: Backend engineer, 6 yrs.
Experience: Senior Engineer, Acme (2020-2024) - Built payments API.
Skills: Python, Go, Postgres, Kafka
Education: BS Computer Science, MIT (2014-2018)'''
profile = asyncio.run(complete_json(_EXTRACTION_SYSTEM, SAMPLE))
import json; print(json.dumps(profile, indent=2)[:800])

## 6. Resume Tailoring (gap analysis)
Given a JD + resume JSON, show matched / missing.

In [ ]:
from agent.resume import tailoring
JD = 'Looking for a backend engineer with Python, Kafka, AWS, and gRPC. Kubernetes preferred.'
sample_profile = {'contact':{'name':'Jane Doe'},'summary':'Backend eng',
    'skills':['Python','Kafka','Postgres'],'experience':[],'education':[],
    'certifications':[],'projects':[]}
gaps = asyncio.run(tailoring.gap_analysis(JD, sample_profile))
print('matched:', gaps['matched'])
print('missing_required:', gaps['missing_required'])
print('missing_preferred:', gaps['missing_preferred'])

## 7. Job Search Tool
Discover roles at Stripe in New York (DuckDuckGo). Network-dependent.

In [ ]:
from agent.tools import company_job_search
res = asyncio.run(company_job_search.run(
    {'company':'Stripe','location':'New York','role':'backend'}, 'test_user'))
for r in res['results'][:3]:
    print(r['ats_platform'], round(r['score'],1), r['title'][:70])

## 8. SSE Event format
Show how each event type serialises.

In [ ]:
import json
EVENTS = [
    {'type':'token','content':'Hello'},
    {'type':'tool_start','tool':'company_job_search'},
    {'type':'tool_end','tool':'company_job_search','result':{}},
    {'type':'progress','step':'Planning'},
    {'type':'hitl_request','action':'write_resume','details':{}},
    {'type':'applied','company':'Stripe','role':'SWE'},
    {'type':'resume_ready','path':'resumes/x.pdf'},
    {'type':'onboarding_required','message':'Upload resume'},
    {'type':'captcha_blocked'},
    {'type':'login_required','url':'https://...'},
    {'type':'done'}]
for e in EVENTS: print('data: ' + json.dumps(e) + chr(10))

## 9. LangSmith
Verify env vars, create a run, log metadata, close it.

In [ ]:
from observability import langsmith
print('tracing enabled:', langsmith.tracing_enabled())
rid = langsmith.create_run('nb_test', {'foo':'bar'}, metadata={'src':'notebook'})
print('run id:', rid)
langsmith.end_run(rid, outputs={'ok': True})
langsmith.flush_trace()

## 10. LangGraph Agent
Minimal invocation with a 'hello' message (no tools).

In [ ]:
from agent.graph import build_graph, initial_state
graph = build_graph()
state = initial_state('test_user','nb-session','Hi there!', memories=[])
# Force no tools so we don't make tool calls in this smoke test.
state['pending_goals'] = []
out = asyncio.run(graph.ainvoke(state, config={'configurable':{'thread_id':'nb-session'}}))
print('final_response:', out.get('final_response'))

## 11. Full Pipeline smoke test
POST /chat/stream against a locally running server (uvicorn api.main:app).
Start the server first, then register + stream.

In [ ]:
import httpx, json
BASE = 'http://localhost:8000'
uname = 'smoke_' + uuid.uuid4().hex[:6]
try:
    tok = httpx.post(f'{BASE}/auth/register', json={'username':uname,
        'email':uname+'@x.com','password':'secret123'}, timeout=30).json()['access_token']
    events = []
    with httpx.stream('POST', f'{BASE}/chat/stream',
        headers={'Authorization': f'Bearer {tok}'},
        json={'message':'hello'}, timeout=120) as r:
        for line in r.iter_lines():
            if line.startswith('data:'):
                events.append(json.loads(line[5:].strip()))
    for e in events: print(e)
except Exception as ex:
    print('Is the server running? Error:', ex)